# OpenAI 답변 생성 시도

10월 2일 OpenAI 생성 실험의 당시 작업본. 기존 EXAONE 경로도 함께 남아 있는 탐색 버전입니다.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


In [ ]:
%pip install -q openai

In [ ]:
%pip install -q langchain-openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

## 준비. 필요한 도구 설치

In [ ]:
%pip install -q numpy faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" "autoawq==0.2.9" accelerate

## 준비. 내 문서 폴더 연결

In [ ]:
import os
import re
import numpy as np
from google.colab import drive
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"


## 1. 메타 데이터 불러오기


원본 데이터와 csv 파일 불러와 함께 저장

메타데이터에 있는 내용(예: 공고 번호, 공고 차수, 사업명, 사업 금액, 발주 기관, 공개 일자, 입찰 참여 시작일, 입찰 참여 마감일)을 물어 볼 수 있음.

In [ ]:
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader

# 1. pandas로 CSV 읽기
df = pd.read_csv("/content/drive/MyDrive/data_list.csv", encoding="utf-8")

In [ ]:
df.columns

이름 정규화를 함

파일을 다운로드 해서 그런지 원본 데이터으 파일명이 깨져있음

In [ ]:
import unicodedata
import re

def normalize_filename(x): # 정규화 하는 함수
    x = str(x)

    # 확장자 제거 (rsplit)
    x = x.rsplit(".", 1)[0]

    # 유니코드 정규화
    x = unicodedata.normalize("NFKC", x)

    # 공백 정리
    x = re.sub(r"\s+", " ", x).strip()

    return x

In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm

from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle

all_files = []
documents = []

for name in os.listdir(folder)[:30]:
    full_path = os.path.join(folder, name)

    # 1. 본문 가져오기
    if name.rsplit(".", 1)[1].lower() == "pdf":
        text = pymupdf4llm.to_markdown(full_path)

    else:
        options = ExtractOptions(
            table_style=TableStyle.MARKDOWN
        )

        loader = HwpHwpxLoader(
            full_path,
            mode="single",
            extract_options=options
        )

        loaded_docs = loader.load()
        text = loaded_docs[0].page_content

    # 2. 메타데이터 찾기
    normalized_name = normalize_filename(name)

    result_df = df[
        df["파일명"].apply(normalize_filename) == normalized_name
    ]

    if not result_df.empty:
        row = result_df.iloc[0]

        metadata = {
            "공고 번호": row["공고 번호"],
            "공고 차수": row["공고 차수"],
            "사업명": row["사업명"],
            "사업 금액": row["사업 금액"],
            "발주 기관": row["발주 기관"],
            "공개 일자": row["공개 일자"],
            "입찰 참여 시작일": row["입찰 참여 시작일"],
            "입찰 참여 마감일": row["입찰 참여 마감일"],
            "파일명": row["파일명"]
        }

        # 전체 메타데이터 모음
        all_files.append(metadata)

        # 현재 문서 + 현재 문서의 metadata
        documents.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

In [ ]:
# 원본 데이터 파일 경로 리스트에 저장
files = []
for name in os.listdir(folder):
    full_path = os.path.join(folder, name)
    files.append(full_path)

## 2. 확장자가 각각 몇 개인지 확인
원본 데이터 불러오기 전에 확장자 확인

확장자에 따른 로더 선택

In [ ]:
extensions = []
for f in files:
    extension = f.rsplit(".", 1)[1].lower()
    extensions.append(extension)

extensions = np.array(extensions)
types, counts = np.unique(extensions, return_counts=True)
for extension, count in zip(types, counts):
    print(extension, count)


## 4. 불러온 내용 확인과 공백 정리


In [ ]:
documents[0].metadata, documents[0].page_content[:500]

In [ ]:
texts = []
for doc in documents:
    text = re.sub(r"[\t+]", " ", doc.page_content)
    texts.append({
        "file": doc.metadata,
        "text": text,
    })


In [ ]:
texts

## 5. 문서 청킹


In [ ]:
separators = ["\n\n", "\n", " "]
splitter = RecursiveCharacterTextSplitter(
    separators=separators, chunk_size=700, chunk_overlap=100
)

chunks = []
for document in texts:
    for piece in splitter.split_text(document["text"]):
        chunks.append({"file": document["file"], "text": piece})

print("청크 수:", len(chunks))
print("첫 청크:\n", chunks[0] if chunks else "없음")


### 6번에서 사용할 임베딩 모델을 고른 이유


## 6. 벡터 DB에 저장


In [ ]:
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

chunk_documents = [
    Document(page_content=c["text"], metadata={"file": c["file"]})
    for c in chunks
]

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=userdata.get("OPENAI_API_KEY"),
)
vector_store = FAISS.from_documents(chunk_documents, embeddings)

print("OpenAI 임베딩으로 저장한 청크:", vector_store.index.ntotal)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="Alibaba-NLP/gte-multilingual-base",
    model_kwargs={"trust_remote_code": True},
)
chunk_documents = [Document(page_content=c["text"], metadata={"file": c["file"]}) for c in chunks]
vector_store = FAISS.from_documents(chunk_documents, embeddings)
print("저장한 청크:", vector_store.index.ntotal)


### 7번에서 사용할 생성 모델 비교


## 7. 모델과 토크나이저 설정


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace


language_model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ"
model_revision = "f2699f35d5b4ba511ab93826c22bd07370296b5c"


model = AutoModelForCausalLM.from_pretrained(
    language_model_name, torch_dtype=torch.float16,
    trust_remote_code=True, device_map="auto", revision=model_revision
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name, revision=model_revision)


llm_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.2,
    return_full_text=False,
    max_new_tokens=220,
)


llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)


## 8. 프롬프트 작성 + 기본 RAG / HyDE


In [ ]:
answer_rules = '''너는 제공된 RFP 문서를 근거로 질문에 답하는 도우미야.
아래 문서에서 질문과 직접 관련된 내용만 사용해 답해.
- 문서에 없는 사실이나 추측을 추가하지 마. 특히 금액, 날짜, 자격 조건을 지어내지 마.
- 대상, 조건, 예외, 시점을 구분해. 허용과 금지를 바꾸어 설명하지 마.
- 질문의 일부만 확인되면 확인된 부분만 답하고, 나머지는 확인할 수 없다고 밝혀.
- 관련 근거가 전혀 없으면 "제공된 문서에서 확인할 수 없습니다."라고만 답해.
- 간결하게 한국어로 답하고, 사용한 문서 번호를 [1]처럼 붙여.

답변 형식 예시 (실제 문서 근거 아님):
문서: [1] 입찰 참여 마감일은 10월 5일이다.
질문: 입찰 참여 마감일은 언제인가요?
답변: 입찰 참여 마감일은 10월 5일입니다. [1]'''

basic_answer_rules = answer_rules
hyde_answer_rules = answer_rules + "\nHyDE의 가상 안내문은 근거가 아니야. 실제 문서 내용만 사용해."

hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

def search(question, method="basic"):
    search_text = question
    if method == "hyde":
        prompt = f"{hyde_search_rules}\n\n질문: {question}\n가상 안내문:"
        search_text = chat_model.invoke(prompt).content.strip()

    found = vector_store.similarity_search(search_text, k=8)
    results = []
    for doc in found:
        results.append({"file": doc.metadata["file"], "text": doc.page_content})
    return results


## 9. 리랭킹


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("Alibaba-NLP/gte-multilingual-reranker-base", trust_remote_code=True)

def rerank(question, found):
    if not found:
        return []

    texts = [item["text"] for item in found]
    ranking = reranker.rank(question, texts, top_k=4)

    return [found[row["corpus_id"]] for row in ranking]


## 10. 질문하기


In [ ]:
questions = [
    "벤처확인종합관리시스템 기능 고도화 용역의 사업기간과 예산은 얼마인가요?",
    "벤처확인종합관리시스템 기능 고도화 용역의 기술평가와 가격평가 비율은 각각 얼마인가요?",
    "BIFF & ACFM 온라인서비스 재개발 사업의 기간과 예산은 얼마인가요?",
    "BIFF & ACFM 온라인서비스 사업에서 BIFF와 ACFM에 각각 어떤 인력을 상주 파견하나요?",
    "KUSF 체육특기자 경기기록 관리시스템은 1차 연도에 어느 종목을 우선하고, 이후 어떤 종목으로 확대하나요?",
    "예술경영지원센터 통합 정보시스템 구축 사전 컨설팅의 제안서는 어떻게 제출하며, 기한이 지난 서류는 인정되나요?",
]


In [ ]:
question = questions[0]
method = "basic"

found = search(question, method)
found = rerank(question, found)
context = "\n\n".join(
    f"[{i+1}] {c['file']}\n{c['text']}"
    for i, c in enumerate(found)
)
instructions = basic_answer_rules if method == "basic" else hyde_answer_rules
answer_prompt = f"{instructions}\n\n실제 문서 내용:\n{context}\n\n실제 질문: {question}\n답변:"
answer = chat_model.invoke(answer_prompt).content.strip()

print(answer)


In [ ]:
print("\n=== context ===")
print(context)

In [ ]:
question = questions[0]
method = "hyde"

found = search(question, method)
found = rerank(question, found)
context = "\n\n".join(
    f"[{i+1}] {c['file']}\n{c['text']}"
    for i, c in enumerate(found)
)
instructions = basic_answer_rules if method == "basic" else hyde_answer_rules
answer_prompt = f"{instructions}\n\n실제 문서 내용:\n{context}\n\n실제 질문: {question}\n답변:"
answer = chat_model.invoke(answer_prompt).content.strip()

print(answer)


In [ ]:
print("\n=== context ===")
print(context)

In [ ]:
question = questions[0]
method = "basic"

found = search(question, method)
found = rerank(question, found)
context = "\n\n".join(
    f"[{i+1}] {c['file']}\n{c['text']}"
    for i, c in enumerate(found)
)
instructions = basic_answer_rules if method == "basic" else hyde_answer_rules
answer_prompt = f"{instructions}\n\n실제 문서 내용:\n{context}\n\n실제 질문: {question}\n답변:"

response = client.responses.create(
    model="gpt-5-mini-2025-08-07",
    input= answer_prompt
)

print(response.output_text)

In [ ]:
question = questions[0]
method = "hyde"

found = search(question, method)
found = rerank(question, found)
context = "\n\n".join(
    f"[{i+1}] {c['file']}\n{c['text']}"
    for i, c in enumerate(found)
)
instructions = basic_answer_rules if method == "basic" else hyde_answer_rules
answer_prompt = f"{instructions}\n\n실제 문서 내용:\n{context}\n\n실제 질문: {question}\n답변:"

response = client.responses.create(
    model="gpt-5-mini-2025-08-07",
    input= answer_prompt
)

print(response.output_text)